In [1]:
import cv2 #opencv-python
import numpy as np
from pathlib import Path
import json
import pandas as pd

In [2]:
def _load_color_ranges(path="color_ranges.json"):
    p = Path(path)
    if p.exists():
        raw = json.loads(p.read_text())
        print(f"Loaded calibrated color ranges from {path}")
    else:
        raw = _DEFAULT_COLOR_RANGES
        print(f"No {path} found, using default color ranges")

    # convert plain lists back into (np.array, np.array) tuples per color
    return {color: [(np.array(lo), np.array(hi)) for lo, hi in ranges] for color, ranges in raw.items()}

In [3]:
def detect_point(frame, color, last_xy=None, roi_radius=80, full_scale_fallback=0.5, annotated=None):
    """
    Detect the largest blob of `color` ("red" or "green") and return its centroid (x, y).
    Draws onto `annotated` if provided (so multiple colors can share one output frame),
    otherwise draws onto a fresh copy of `frame`.
    Returns:
        (x, y, mask, annotated_frame)
        x, y are None if no object of that color is detected.
    """
    if annotated is None:
        annotated = frame.copy()

    h, w = frame.shape[:2]
    ranges = _COLOR_RANGES[color]
    draw_color = _COLOR_DRAW[color]

    def _mask_and_contours(img):
        hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
        m = None
        for lo, hi in ranges:
            part = cv2.inRange(hsv, lo, hi)
            m = part if m is None else cv2.bitwise_or(m, part)
        m = cv2.morphologyEx(m, cv2.MORPH_OPEN, _kernel)
        m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, _kernel)
        cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        return m, cnts

    def _centroid(cnt):
        M = cv2.moments(cnt)
        if M["m00"] == 0:
            return None
        return M["m10"] / M["m00"], M["m01"] / M["m00"]

    def _draw(cx, cy, cnt):
        cv2.circle(annotated, (cx, cy), 6, draw_color, -1)
        cv2.drawContours(annotated, [cnt], -1, draw_color, 2)
        cv2.putText(annotated, f"{color}({cx}, {cy})", (cx + 10, cy - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, draw_color, 1, cv2.LINE_AA)

    #search only near the last known point, at full resolution
    if last_xy is not None:
        lx, ly = last_xy
        x0, x1 = max(0, lx - roi_radius), min(w, lx + roi_radius)
        y0, y1 = max(0, ly - roi_radius), min(h, ly + roi_radius)
        crop = frame[y0:y1, x0:x1]

        mask_small, contours = _mask_and_contours(crop)
        if contours:
            largest = max(contours, key=cv2.contourArea)
            if cv2.contourArea(largest) >= 10:
                c = _centroid(largest)
                if c:
                    cx, cy = int(c[0]) + x0, int(c[1]) + y0
                    mask = np.zeros((h, w), dtype=np.uint8)
                    mask[y0:y1, x0:x1] = mask_small
                    _draw(cx, cy, largest + [x0, y0])
                    return cx, cy, mask, annotated
        #fall through to full-frame search if nothing found in the ROI

    #fallback: downscaled full-frame search (re-acquire track)
    small = cv2.resize(frame, None, fx=full_scale_fallback, fy=full_scale_fallback, interpolation=cv2.INTER_AREA)
    mask_small, contours = _mask_and_contours(small)
    mask = cv2.resize(mask_small, (w, h), interpolation=cv2.INTER_NEAREST)

    if not contours:
        return None, None, mask, annotated

    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 10 * (full_scale_fallback ** 2):
        return None, None, mask, annotated

    c = _centroid(largest)
    if c is None:
        return None, None, mask, annotated

    cx, cy = int(c[0] / full_scale_fallback), int(c[1] / full_scale_fallback)
    largest_scaled = (largest.astype(np.float32) / full_scale_fallback).astype(np.int32)
    _draw(cx, cy, largest_scaled)
    return cx, cy, mask, annotated

In [ ]:
def process_video(video_path, output_csv, preview=True, save_annotated=False):
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    writer = None
    if save_annotated:
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        annotated_path = video_path.with_name(video_path.stem + "_tracked.mp4")
        writer = cv2.VideoWriter(str(annotated_path), fourcc, fps, (width, height))

    results = []
    frame_idx = 0
    last_xy_red = None
    last_xy_green = None

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        x_r, y_r, mask_r, annotated = detect_point(frame, "red", last_xy=last_xy_red)
        x_g, y_g, mask_g, annotated = detect_point(frame, "green", last_xy=last_xy_green, annotated=annotated)

        if x_r is not None:
            last_xy_red = (x_r, y_r)
        if x_g is not None:
            last_xy_green = (x_g, y_g)

        time_s = frame_idx / fps if fps > 0 else np.nan

        results.append({
            "frame": frame_idx,
            "time_s": time_s,
            "x_red": x_r,
            "y_red": y_r,
            "x_green": x_g,
            "y_green": y_g
        })

        if writer is not None:
            writer.write(annotated)

        if preview:
            combined_mask = cv2.cvtColor(cv2.bitwise_or(mask_r, mask_g), cv2.COLOR_GRAY2BGR)
            top = np.hstack((frame, annotated))
            bottom = np.hstack((combined_mask, np.zeros_like(combined_mask)))
            display = np.vstack((top, bottom))

            scale = 0.7
            display = cv2.resize(display, None, fx=scale, fy=scale)

            cv2.imshow("Original | Annotated / Mask", display)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

        frame_idx += 1

        if total_frames > 0 and (frame_idx % 5 == 0 or frame_idx == total_frames):
            percent = (frame_idx / total_frames) * 100
            print(f"\rProcessing: {percent:.2f}%", end="", flush=True)

    print()
    cap.release()
    if writer is not None:
        writer.release()
    cv2.destroyAllWindows()

    df = pd.DataFrame(results)
    df.to_csv(output_csv, index=False)

    print(f"Done. Saved coordinates to: {output_csv}")
    if save_annotated:
        print(f"Saved annotated video to: {annotated_path}")

    missing_r = df["x_red"].isna().sum()
    missing_g = df["x_green"].isna().sum()
    print(f"Frames processed: {len(df)}")
    print(f"Frames with missing red detection: {missing_r}")
    print(f"Frames with missing green detection: {missing_g}")

    #add feature that finds points that should be blocked, by finding if missing green points are in the rectangle between blue and red

In [ ]:
def export_first_frame(video_path, output_png=None, dpi=300):
    """
    Export the first frame of a video as a PNG.

    The exported image keeps exactly the same pixel width and height
    as the original video frame.

    Parameters
    ----------
    video_path : str or Path
        Path to the input video.

    output_png : str or Path, optional
        Output PNG path. If None, the output name will be:
        <video_name>_first_frame.png

    dpi : float
        DPI metadata written to the PNG.
        DPI does not change the number of pixels or image quality.
    """

    video_path = Path(video_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video file not found: {video_path}")

    if output_png is None:
        output_png = video_path.with_name(video_path.stem + "_first_frame.png")
    else:
        output_png = Path(output_png)

    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    success, frame = cap.read()
    cap.release()

    if not success or frame is None:
        raise RuntimeError("Could not read the first frame from the video.")

    height, width = frame.shape[:2]

    #OpenCV writes the original pixel data without resizing.
    success = cv2.imwrite(str(output_png), frame, [cv2.IMWRITE_PNG_COMPRESSION, 3])

    if not success:
        raise RuntimeError(f"Could not save PNG: {output_png}")

    #OpenCV does not reliably write DPI metadata, so add it using Pillow.
    try:
        from PIL import Image

        with Image.open(output_png) as image:
            image.save(output_png, dpi=(dpi, dpi))

    except ImportError:
        print("Pillow is not installed, so DPI metadata was not added.")
        print("Install it with: pip install pillow")

    print(f"Saved first frame to: {output_png}")
    print(f"Resolution: {width} x {height} pixels")
    print(f"DPI metadata: {dpi} DPI")

    return output_png

In [6]:
_kernel = np.ones((5, 5), np.uint8)

_DEFAULT_COLOR_RANGES = {
    "red": [
        [[0, 130, 90], [8, 255, 255]],
        [[172, 130, 90], [180, 255, 255]],
    ],
    "green": [
        [[45, 100, 60], [75, 255, 255]],
    ],
}

_COLOR_RANGES = _load_color_ranges()

_COLOR_DRAW = {
    "red": (0, 255, 0),
    "green": (0, 0, 255),
}

video = "IMG_6437.MOV"
output = "20260710_141408.csv"

process_video(video_path=video, output_csv=output, preview=True, save_annotated=True)

export_first_frame(video_path=video, output_png="20260710_141408_first_frame.png", dpi=300)

Loaded calibrated color ranges from color_ranges.json


FileNotFoundError: Video file not found: IMG_6437.MOV